In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Imports

import os
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torchvision import models, transforms

In [3]:
# Device

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cuda


In [4]:
# Paths

CNN_MODEL_PATH = "/content/drive/MyDrive/EMBED/models/best_resnet50_embed_5yr_risk.pth"
LSTM_MODEL_PATH = "/content/drive/MyDrive/EMBED/models/best_lstm_future_risk.pth"

In [5]:
# Load trained ResNet50 future-risk model as CNN feature extractor

cnn_model = models.resnet50(weights=None)

num_features = cnn_model.fc.in_features

cnn_model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(num_features, 5)
)

cnn_model.load_state_dict(
    torch.load(CNN_MODEL_PATH, map_location=device)
)

cnn_model = cnn_model.to(device)
cnn_model.eval()

feature_extractor = nn.Sequential(
    *list(cnn_model.children())[:-1]
)

feature_extractor = feature_extractor.to(device)
feature_extractor.eval()

print("CNN feature extractor loaded.")
print("Feature size:", num_features)

CNN feature extractor loaded.
Feature size: 2048


In [6]:
class FutureRiskLSTM(nn.Module):

    def __init__(
        self,
        input_size=6149,
        compressed_size=256,
        hidden_size=128,
        num_layers=1,
        output_size=5,
        dropout=0.3
    ):
        super(FutureRiskLSTM, self).__init__()

        self.feature_compressor = nn.Sequential(
            nn.Linear(input_size, compressed_size),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.lstm = nn.LSTM(
            input_size=compressed_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.output_head = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, output_size)
        )

    def forward(self, x, mask):
        x = self.feature_compressor(x)

        lstm_output, _ = self.lstm(x)

        lengths = mask.sum(dim=1).long()
        last_indices = lengths - 1

        batch_indices = torch.arange(
            x.size(0),
            device=x.device
        )

        last_outputs = lstm_output[
            batch_indices,
            last_indices
        ]

        logits = self.output_head(last_outputs)

        return logits

In [7]:
lstm_model = FutureRiskLSTM().to(device)

lstm_model.load_state_dict(
    torch.load(LSTM_MODEL_PATH, map_location=device)
)

lstm_model.eval()

print("LSTM model loaded successfully.")

LSTM model loaded successfully.


In [8]:
# Image transform

inference_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [9]:
# Extract one CNN feature vector from one image

def extract_image_feature(image_path):
    image = Image.open(image_path).convert("RGB")
    image = inference_transform(image)
    image = image.unsqueeze(0).to(device)

    with torch.no_grad():
        feature = feature_extractor(image)
        feature = feature.view(feature.size(0), -1)

    return feature.cpu().numpy().squeeze()

In [10]:
test_image_path = "/content/drive/MyDrive/EMBED/processed_images_512/92802516_9905843998586960_R_MLO_909.png"

test_feature = extract_image_feature(test_image_path)

print(test_feature.shape)

(2048,)


In [11]:
# Build one session feature from uploaded view image paths

def build_session_feature(session, recency_weight=1.0):
    view_slots = ["L_CC", "R_CC", "L_MLO", "R_MLO"]

    view_features = {}
    view_mask = []

    for slot in view_slots:
        image_path = session.get(slot)

        if image_path is None:
            view_features[slot] = None
            view_mask.append(0)
        else:
            view_features[slot] = extract_image_feature(image_path)
            view_mask.append(1)

    available_features = [
        feature for feature in view_features.values()
        if feature is not None
    ]

    if len(available_features) == 0:
        raise ValueError("Session must contain at least one valid image.")

    exam_feature = np.mean(available_features, axis=0)

    zero_feature = np.zeros(2048)

    if view_features["L_CC"] is not None and view_features["R_CC"] is not None:
        cc_asymmetry = np.abs(view_features["L_CC"] - view_features["R_CC"])
    else:
        cc_asymmetry = zero_feature

    if view_features["L_MLO"] is not None and view_features["R_MLO"] is not None:
        mlo_asymmetry = np.abs(view_features["L_MLO"] - view_features["R_MLO"])
    else:
        mlo_asymmetry = zero_feature

    final_session_feature = np.concatenate([
        exam_feature,
        cc_asymmetry,
        mlo_asymmetry,
        np.array(view_mask),
        np.array([recency_weight])
    ])

    return final_session_feature

In [12]:
sample_session = {
    "exam_date": "2020-01-15",
    "L_CC": "/content/drive/MyDrive/EMBED/processed_images_512/92802516_9905843998586960_L_CC_910.png",
    "R_CC": None,
    "L_MLO": None,
    "R_MLO": None
}

sample_feature = build_session_feature(sample_session)

print(sample_feature.shape)

(6149,)


In [13]:
def calculate_recency_weights_for_sessions(sessions):
    exam_dates = [
        pd.to_datetime(session["exam_date"])
        for session in sessions
    ]

    latest_date = max(exam_dates)

    weights = []

    for exam_date in exam_dates:
        years_before_latest = (
            latest_date - exam_date
        ).days / 365.25

        weight = np.exp(-0.5 * years_before_latest)
        weights.append(weight)

    weights = np.array(weights)
    weights = weights / weights.sum()

    return weights

In [14]:
def get_age_weight(age):
    if age is None:
        return 1.0

    if age < 40:
        return 0.90
    elif age < 50:
        return 1.00
    elif age < 60:
        return 1.10
    elif age < 70:
        return 1.20
    else:
        return 1.30

In [15]:

def session_has_image(session):
    view_slots = ["L_CC", "R_CC", "L_MLO", "R_MLO"]

    return any(
        session.get(slot) is not None
        for slot in view_slots
    )


def prepare_sessions(sessions, max_sessions=5):
    # Remove empty sessions, sort by exam date, and keep only the latest sessions.
    valid_sessions = [
        session.copy()
        for session in sessions
        if session.get("exam_date") is not None and session_has_image(session)
    ]

    if len(valid_sessions) == 0:
        raise ValueError("At least one session with one valid image is required.")

    valid_sessions = sorted(
        valid_sessions,
        key=lambda x: pd.to_datetime(x["exam_date"])
    )

    if len(valid_sessions) > max_sessions:
        valid_sessions = valid_sessions[-max_sessions:]

    return valid_sessions


def predict_patient_future_risk(sessions, patient_age=None, max_sessions=5):
    # Frontend supports up to 5 sessions, so we use the latest 5 sessions only.
    sessions = prepare_sessions(
        sessions,
        max_sessions=max_sessions
    )

    recency_weights = calculate_recency_weights_for_sessions(
        sessions
    )

    session_features = []

    for session, weight in zip(sessions, recency_weights):
        feature = build_session_feature(
            session,
            recency_weight=weight
        )

        session_features.append(feature)

    sequence = np.stack(
        session_features
    ).astype(np.float32)

    sequence_mask = np.ones(
        sequence.shape[0],
        dtype=np.float32
    )

    sequence_tensor = torch.tensor(
        sequence,
        dtype=torch.float32
    ).unsqueeze(0).to(device)

    mask_tensor = torch.tensor(
        sequence_mask,
        dtype=torch.float32
    ).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = lstm_model(
            sequence_tensor,
            mask_tensor
        )

        probs = torch.sigmoid(
            logits
        ).cpu().numpy().squeeze()

    age_weight = get_age_weight(
        patient_age
    )

    adjusted_probs = np.clip(
        probs * age_weight,
        0,
        1
    )

    return {
        "risk_1yr": float(adjusted_probs[0]),
        "risk_2yr": float(adjusted_probs[1]),
        "risk_3yr": float(adjusted_probs[2]),
        "risk_4yr": float(adjusted_probs[3]),
        "risk_5yr": float(adjusted_probs[4]),
        "age_weight": float(age_weight),
        "num_sessions_used": int(len(sessions))
    }


In [16]:
def risk_category(risk_value):
    if risk_value < 0.30:
        return "Low"
    elif risk_value < 0.60:
        return "Moderate"
    else:
        return "High"

In [17]:
def format_prediction_output(result):
    final_5yr_risk = result["risk_5yr"]

    return {
        "age_weight": result["age_weight"],
        "risk_predictions": {
            "1_year": f"{result['risk_1yr'] * 100:.2f}%",
            "2_year": f"{result['risk_2yr'] * 100:.2f}%",
            "3_year": f"{result['risk_3yr'] * 100:.2f}%",
            "4_year": f"{result['risk_4yr'] * 100:.2f}%",
            "5_year": f"{result['risk_5yr'] * 100:.2f}%"
        },
        "risk_category": risk_category(final_5yr_risk)
    }

In [18]:
metadata = pd.read_csv(
    "/content/drive/MyDrive/EMBED/embed_cnn_feature_metadata.csv"
)

metadata["study_date_anon"] = pd.to_datetime(
    metadata["study_date_anon"]
)

In [19]:
patient_exam_counts = (
    metadata.groupby("empi_anon")["study_date_anon"]
    .nunique()
    .sort_values(ascending=False)
)

patient_exam_counts.head(10)

,study_date_anon
empi_anon,
12249159,13
64701315,13
29064737,12
55362005,11
92802516,11
70180258,11
69190206,9
59795563,9
32688964,8


In [20]:
patient_id = patient_exam_counts.index[0]

print("Patient:", patient_id)

patient_df = metadata[
    metadata["empi_anon"] == patient_id
]

patient_df[
    [
        "study_date_anon",
        "ImageLateralityFinal",
        "ViewPosition",
        "processed_image_path"
    ]
].sort_values("study_date_anon")

Patient: 12249159


,study_date_anon,ImageLateralityFinal,ViewPosition,processed_image_path
560,2013-02-22,R,CC,/content/drive/MyDrive/EMBED/processed_images_...
571,2013-02-22,R,MLO,/content/drive/MyDrive/EMBED/processed_images_...
566,2013-02-22,L,CC,/content/drive/MyDrive/EMBED/processed_images_...
562,2013-02-22,L,MLO,/content/drive/MyDrive/EMBED/processed_images_...
553,2014-02-28,R,MLO,/content/drive/MyDrive/EMBED/processed_images_...
568,2014-02-28,L,MLO,/content/drive/MyDrive/EMBED/processed_images_...
556,2014-02-28,R,CC,/content/drive/MyDrive/EMBED/processed_images_...
552,2014-02-28,L,CC,/content/drive/MyDrive/EMBED/processed_images_...
545,2014-04-11,R,CC,/content/drive/MyDrive/EMBED/processed_images_...
526,2015-01-30,L,CC,/content/drive/MyDrive/EMBED/processed_images_...


In [21]:
sessions = []

for exam_date, exam_df in patient_df.groupby("study_date_anon"):

    session = {
        "exam_date": str(exam_date.date()),
        "L_CC": None,
        "R_CC": None,
        "L_MLO": None,
        "R_MLO": None
    }

    for _, row in exam_df.iterrows():

        slot = (
            row["ImageLateralityFinal"]
            + "_"
            + row["ViewPosition"]
        )

        if slot in session and session[slot] is None:
            session[slot] = row["processed_image_path"]

    sessions.append(session)

print("Number of sessions:", len(sessions))

Number of sessions: 13


In [22]:
for i, session in enumerate(sessions[:3]):
    print(f"\nSession {i+1}")
    print(session)


Session 1
{'exam_date': '2013-02-22', 'L_CC': '/content/drive/MyDrive/EMBED/processed_images_512/12249159_7848876025902862_L_CC_566.png', 'R_CC': '/content/drive/MyDrive/EMBED/processed_images_512/12249159_7848876025902862_R_CC_560.png', 'L_MLO': '/content/drive/MyDrive/EMBED/processed_images_512/12249159_7848876025902862_L_MLO_562.png', 'R_MLO': '/content/drive/MyDrive/EMBED/processed_images_512/12249159_7848876025902862_R_MLO_571.png'}

Session 2
{'exam_date': '2014-02-28', 'L_CC': '/content/drive/MyDrive/EMBED/processed_images_512/12249159_9529855822612122_L_CC_552.png', 'R_CC': '/content/drive/MyDrive/EMBED/processed_images_512/12249159_9529855822612122_R_CC_556.png', 'L_MLO': '/content/drive/MyDrive/EMBED/processed_images_512/12249159_9529855822612122_L_MLO_568.png', 'R_MLO': '/content/drive/MyDrive/EMBED/processed_images_512/12249159_9529855822612122_R_MLO_553.png'}

Session 3
{'exam_date': '2014-04-11', 'L_CC': None, 'R_CC': '/content/drive/MyDrive/EMBED/processed_images_512/12

In [23]:
raw_result = predict_patient_future_risk(
    sessions=sessions,
    patient_age=55
)

formatted_result = format_prediction_output(
    raw_result
)

formatted_result

{'age_weight': 1.1,
 'risk_predictions': {'1_year': '9.61%',
  '2_year': '27.45%',
  '3_year': '29.08%',
  '4_year': '34.21%',
  '5_year': '36.42%'},
 'risk_category': 'Moderate'}

In [24]:

def calculate_image_contributions(sessions, patient_age=None, max_sessions=5):
    # Use the same session limit as the main prediction function.
    sessions = prepare_sessions(
        sessions,
        max_sessions=max_sessions
    )

    original_result = predict_patient_future_risk(
        sessions=sessions,
        patient_age=patient_age,
        max_sessions=max_sessions
    )

    risk_keys = [
        "risk_1yr",
        "risk_2yr",
        "risk_3yr",
        "risk_4yr",
        "risk_5yr"
    ]

    contribution_rows = []

    for session_idx, session in enumerate(sessions):
        available_views = [
            view
            for view in ["L_CC", "R_CC", "L_MLO", "R_MLO"]
            if session.get(view) is not None
        ]

        # If a session only has one image, removing it would make that session empty.
        # So we skip ablation for that session.
        if len(available_views) <= 1:
            continue

        for view_slot in available_views:
            modified_sessions = [
                s.copy()
                for s in sessions
            ]

            modified_sessions[session_idx][view_slot] = None

            modified_result = predict_patient_future_risk(
                sessions=modified_sessions,
                patient_age=patient_age,
                max_sessions=max_sessions
            )

            row = {
                "session": int(session_idx + 1),
                "exam_date": str(session["exam_date"]),
                "view": view_slot
            }

            for key in risk_keys:
                contribution = (
                    original_result[key]
                    - modified_result[key]
                )

                row[key + "_contribution"] = float(contribution)
                row[key + "_contribution_percent_points"] = float(contribution * 100)

            contribution_rows.append(row)

    contribution_df = pd.DataFrame(contribution_rows)

    # Add relative contribution percentages for frontend charts.
    # These show how much each image explains of the total absolute contribution for each year.
    for key in risk_keys:
        col = key + "_contribution"
        percent_col = key + "_relative_contribution_percent"

        if len(contribution_df) == 0:
            continue

        total_abs = contribution_df[col].abs().sum()

        if total_abs == 0:
            contribution_df[percent_col] = 0.0
        else:
            contribution_df[percent_col] = (
                contribution_df[col].abs() / total_abs * 100
            )

    return contribution_df


In [25]:
contribution_df = calculate_image_contributions(
    sessions=sessions,
    patient_age=55
)

contribution_df

,session,exam_date,view,risk_1yr_contribution,risk_1yr_contribution_percent_points,risk_2yr_contribution,risk_2yr_contribution_percent_points,risk_3yr_contribution,risk_3yr_contribution_percent_points,risk_4yr_contribution,risk_4yr_contribution_percent_points,risk_5yr_contribution,risk_5yr_contribution_percent_points,risk_1yr_relative_contribution_percent,risk_2yr_relative_contribution_percent,risk_3yr_relative_contribution_percent,risk_4yr_relative_contribution_percent,risk_5yr_relative_contribution_percent
0,1,2018-02-23,L_CC,-0.003868,-0.386830,-0.005607,-0.560707,-0.057917,-5.791721,-0.058670,-5.867022,-0.082364,-8.236361,3.903637,7.190582,5.820470,5.595531,6.071720
1,1,2018-02-23,R_CC,-0.003565,-0.356482,-0.005247,-0.524715,-0.055801,-5.580109,-0.056674,-5.667418,-0.079535,-7.953453,3.597380,6.729012,5.607808,5.405163,5.863165
2,1,2018-02-23,L_MLO,-0.005913,-0.591270,-0.009467,-0.946680,-0.091145,-9.114534,-0.092194,-9.219375,-0.129229,-12.922850,5.966704,12.140351,9.159777,8.792757,9.526529
3,1,2018-02-23,R_MLO,-0.005370,-0.536956,-0.008803,-0.880259,-0.085126,-8.512583,-0.086359,-8.635917,-0.121120,-12.112027,5.418604,11.288566,8.554839,8.236298,8.928803
4,2,2019-03-01,L_CC,-0.005438,-0.543763,-0.006561,-0.656098,-0.076038,-7.603768,-0.077969,-7.796949,-0.108377,-10.837656,5.487301,8.413892,7.641512,7.436152,7.989356
5,2,2019-03-01,R_CC,-0.005183,-0.518277,-0.006402,-0.640178,-0.075282,-7.528177,-0.077346,-7.734600,-0.107613,-10.761282,5.230104,8.209727,7.565546,7.376688,7.933054
6,2,2019-03-01,L_MLO,-0.006782,-0.678239,-0.009411,-0.941110,-0.094357,-9.435716,-0.096037,-9.603739,-0.134111,-13.411099,6.844339,12.068919,9.482554,9.159334,9.886459
7,2,2019-03-01,R_MLO,-0.006422,-0.642172,-0.008870,-0.886974,-0.090498,-9.049755,-0.092367,-9.236744,-0.128906,-12.890622,6.480370,11.374673,9.094677,8.809322,9.502771
8,3,2019-04-20,R_CC,0.000244,0.024424,-0.000111,-0.011143,-0.004134,-0.413433,-0.004204,-0.420395,-0.006148,-0.614792,0.246468,0.142900,0.415485,0.400941,0.453215
9,3,2019-04-20,R_MLO,-0.000363,-0.036311,-0.000088,-0.008789,0.003812,0.381193,0.003960,0.395977,0.005673,0.567260,0.366428,0.112708,0.383085,0.377654,0.418176


In [28]:
def calculate_session_contributions(contribution_df):
    session_columns = [
        "risk_1yr_relative_contribution_percent",
        "risk_2yr_relative_contribution_percent",
        "risk_3yr_relative_contribution_percent",
        "risk_4yr_relative_contribution_percent",
        "risk_5yr_relative_contribution_percent"
    ]

    session_contribution_df = (
        contribution_df
        .groupby(["session", "exam_date"])[session_columns]
        .sum()
        .reset_index()
        .round(2)
    )

    return session_contribution_df

In [30]:

def frontend_ready_prediction(sessions, patient_age=None, max_sessions=5):
    # Keep frontend behaviour consistent: use a maximum of 5 most recent sessions.
    sessions_used = prepare_sessions(
        sessions,
        max_sessions=max_sessions
    )

    raw_result = predict_patient_future_risk(
        sessions=sessions_used,
        patient_age=patient_age,
        max_sessions=max_sessions
    )

    formatted_result = format_prediction_output(
        raw_result
    )

    contribution_df = calculate_image_contributions(
        sessions=sessions_used,
        patient_age=patient_age,
        max_sessions=max_sessions
    )
    session_contribution_df = calculate_session_contributions(
    contribution_df
    )

    session_contribution_records = session_contribution_df.to_dict(
        orient="records"
    )

    contribution_columns = [
    "session",
    "exam_date",
    "view",
    "risk_1yr_relative_contribution_percent",
    "risk_2yr_relative_contribution_percent",
    "risk_3yr_relative_contribution_percent",
    "risk_4yr_relative_contribution_percent",
    "risk_5yr_relative_contribution_percent"
    ]

    contribution_records = (
          contribution_df[contribution_columns]
          .round(2)
          .to_dict(orient="records")
      )

    sessions_summary = []

    for i, session in enumerate(sessions_used, start=1):
        available_views = [
            view
            for view in ["L_CC", "R_CC", "L_MLO", "R_MLO"]
            if session.get(view) is not None
        ]

        sessions_summary.append({
            "session": int(i),
            "exam_date": str(session["exam_date"]),
            "available_views": available_views,
            "num_available_views": int(len(available_views))
        })

    return {
        "patient_age": patient_age,
        "age_weight": float(raw_result["age_weight"]),
        "num_sessions_used": int(raw_result["num_sessions_used"]),
        "max_sessions_allowed": int(max_sessions),
        "sessions_used": sessions_summary,
        "risk_predictions": formatted_result["risk_predictions"],
        #"risk_scores_raw": {
          #  "1_year": float(raw_result["risk_1yr"]),
          #  "2_year": float(raw_result["risk_2yr"]),
          #  "3_year": float(raw_result["risk_3yr"]),
          #  "4_year": float(raw_result["risk_4yr"]),
          #  "5_year": float(raw_result["risk_5yr"])
        #},
        "session_contributions": session_contribution_records,
        "risk_category": formatted_result["risk_category"],
        "image_contributions": contribution_records
    }


In [31]:
frontend_output = frontend_ready_prediction(
    sessions=sessions,
    patient_age=55
)

frontend_output

{'patient_age': 55,
 'age_weight': 1.1,
 'num_sessions_used': 5,
 'max_sessions_allowed': 5,
 'sessions_used': [{'session': 1,
   'exam_date': '2018-02-23',
   'available_views': ['L_CC', 'R_CC', 'L_MLO', 'R_MLO'],
   'num_available_views': 4},
  {'session': 2,
   'exam_date': '2019-03-01',
   'available_views': ['L_CC', 'R_CC', 'L_MLO', 'R_MLO'],
   'num_available_views': 4},
  {'session': 3,
   'exam_date': '2019-04-20',
   'available_views': ['R_CC', 'R_MLO'],
   'num_available_views': 2},
  {'session': 4,
   'exam_date': '2019-10-25',
   'available_views': ['R_CC', 'R_MLO'],
   'num_available_views': 2},
  {'session': 5,
   'exam_date': '2020-04-20',
   'available_views': ['L_CC', 'R_CC', 'L_MLO', 'R_MLO'],
   'num_available_views': 4}],
 'risk_predictions': {'1_year': '9.61%',
  '2_year': '27.45%',
  '3_year': '29.08%',
  '4_year': '34.21%',
  '5_year': '36.42%'},
 'session_contributions': [{'session': 1,
   'exam_date': '2018-02-23',
   'risk_1yr_relative_contribution_percent': 1